# Vérifier les gradients et adapter avec LoRA

Modèle E[id]W, corpus original, même LCG que TypeScript. Exécution Python/PyTorch CPU ; vérifier la perte de domaine et celle du corpus initial. Un bigramme sans attention, pas un LLM moderne.


In [ ]:
import math, torch
from torch import nn
torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
train_text = ["le chat voit la lune", "le chien voit la lune",
              "le chat voit la lune", "le chien voit le chat"]
domain_text = ["le chat voit le chien", "le chat voit le chien", "le chien voit le chat"]
vocab = ["<début>", "<fin>"] + list(dict.fromkeys(" ".join(train_text).split()))
def pairs(lines):
    documents = [[0] + [vocab.index(w) for w in line.split()] + [1] for line in lines]
    values = [(doc[i], doc[i+1]) for doc in documents for i in range(len(doc)-1)]
    return torch.tensor([a for a,b in values]), torch.tensor([b for a,b in values])
x, y = pairs(train_text)
xd, yd = pairs(domain_text)
V, D = len(vocab), 4
state = 7
def random_value():
    global state
    state = (1664525*state+1013904223) % 2**32
    return state/2**32
E = torch.tensor([[(random_value()-.5)*.4 for _ in range(D)] for _ in range(V)], requires_grad=True)
W = torch.tensor([[(random_value()-.5)*.4 for _ in range(V)] for _ in range(D)], requires_grad=True)
def objective(e, w, inputs=x, targets=y):
    return nn.functional.cross_entropy(e[inputs] @ w, targets)
print("Modèle bigramme :", V, "tokens ;", 2*V*D, "paramètres ; perte", objective(E,W).item())


In [ ]:
# Vérification indépendante : formule analytique, autograd et différence finie.
L = objective(E,W)
gE, gW = torch.autograd.grad(L, (E,W))
with torch.no_grad():
    dz = torch.softmax(E[x] @ W, dim=-1)
    dz[torch.arange(len(y)), y] -= 1
    dz /= len(y)
    analytic_W = E[x].T @ dz
    analytic_E = torch.zeros_like(E)
    analytic_E.index_add_(0, x, dz @ W.T)
    torch.testing.assert_close(gE, analytic_E, rtol=0, atol=1e-12)
    torch.testing.assert_close(gW, analytic_W, rtol=0, atol=1e-12)
    eps = 1e-5
    plus, minus = W.clone(), W.clone()
    plus[0,0] += eps
    minus[0,0] -= eps
    numeric = (objective(E,plus)-objective(E,minus))/(2*eps)
    assert abs(numeric.item()-gW[0,0].item()) < 1e-8
print("Gradient W[0,0] :", gW[0,0].item(), "; différence finie :", numeric.item(), "; OK")


In [ ]:
optimizer = torch.optim.SGD([E,W], lr=0.8)
initial_loss = objective(E,W).item()
for step in range(100):
    optimizer.zero_grad()
    objective(E,W).backward()
    nn.utils.clip_grad_norm_([E,W], 5.0)
    optimizer.step()
assert objective(E,W).item() < initial_loss
base_E, base_W = E.detach().clone(), W.detach().clone()
print("Base après 100 pas :", objective(base_E,base_W).item())


## Adapter la projection, base gelée

A initialisé et B nul. Alpha=r ; rang 1. Pas d’adaptation de E, pas d’optimisation de W en mode LoRA. Les assertions vérifient gradient initial et immutabilité.


In [ ]:
# Convention activations lignes : delta_W = A @ B ; alpha/r=1.
rank = 1
state = 11
A = torch.tensor([[(random_value()-.5)*.4 for _ in range(rank)] for _ in range(D)], requires_grad=True)
B = torch.zeros(rank,V, requires_grad=True)
before_domain = objective(base_E,base_W,xd,yd).item()
ga, gb = torch.autograd.grad(objective(base_E,base_W+A@B,xd,yd), (A,B))
assert torch.count_nonzero(ga) == 0 and torch.count_nonzero(gb) > 0
optimizer_lora = torch.optim.SGD([A,B], lr=1.0)
saved_E, saved_W = base_E.clone(), base_W.clone()
for step in range(150):
    optimizer_lora.zero_grad()
    objective(base_E,base_W+A@B,xd,yd).backward()
    nn.utils.clip_grad_norm_([A,B], 5.0)
    optimizer_lora.step()
torch.testing.assert_close(base_E,saved_E,rtol=0,atol=0)
torch.testing.assert_close(base_W,saved_W,rtol=0,atol=0)
after_domain = objective(base_E,base_W+A@B,xd,yd).item()
assert after_domain < before_domain
print("LoRA :", rank*(D+V), "paramètres ; domaine avant/après", before_domain, after_domain)
print("Perte corpus initial après LoRA :", objective(base_E,base_W+A@B).item())
print("Poids de base conservés exactement : OK")


In [ ]:
full_E = base_E.clone().requires_grad_()
full_W = base_W.clone().requires_grad_()
optimizer_full = torch.optim.SGD([full_E,full_W], lr=0.4)
for step in range(150):
    optimizer_full.zero_grad()
    objective(full_E,full_W,xd,yd).backward()
    nn.utils.clip_grad_norm_([full_E,full_W],5.0)
    optimizer_full.step()
print("Fine-tuning complet :", 2*V*D, "paramètres")
print("Perte domaine :", objective(full_E,full_W,xd,yd).item())
print("Perte corpus initial :", objective(full_E,full_W).item())
assert objective(full_E,full_W,xd,yd).item() < before_domain
# Les deux expériences ont des taux explicites différents.
# Pour une étude comparative, explorer les taux et utiliser un test final distinct.


## Interpréter

Les pertes viennent du code exécuté. Même budget de 150 pas, mais taux LoRA=1 et complet=0,4 : ne pas déduire une supériorité générale. Répéter avec un protocole plus large et un jeu final indépendant.

Sources : [LoRA, article original](https://arxiv.org/abs/2106.09685), [PyTorch Autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).
